[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/28_moe_interview.ipynb)

# 🔴 Solution: Mixture of Experts (MoE)（面试版）

## 📋 题目描述

**难度：** Hard

实现混合专家（MoE）层（nn.Module）。

MoE 通过学习的路由器将每个 token 路由到 top-k 个专家，用 softmax 归一化的权重组合它们的输出，实现条件计算。

**签名:** `MixtureOfExperts(d_model, d_ff, num_experts, top_k=2)`（nn.Module）

**前向传播:** `forward(x) -> Tensor`
- `x` — 输入张量 (B, S, d_model)

**返回:** 输出张量 (B, S, d_model)

**约束:**
- 路由器：`nn.Linear(d_model, num_experts)` -> topk -> softmax
- 每个专家：Linear -> ReLU -> Linear
- 专家存储在 `self.experts`（nn.ModuleList）中

**提示：** `router`：`Linear(d, num_experts)` → `topk` → `softmax` 权重。每个专家：`Linear → ReLU → Linear`。对每个 token 的 top-k 专家输出加权求和。


In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


In [ ]:
# ✅ INTERVIEW

class MixtureOfExperts(nn.Module):
    def __init__(self, d_model, d_ff, num_experts, top_k=2):
        super().__init__()
        self.top_k = top_k
        self.router = nn.Linear(d_model, num_experts)
        self.experts = nn.ModuleList([
            nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
            for _ in range(num_experts)
        ])

    def forward(self, x):
        orig_shape = x.shape
        if x.dim() == 3:
            B, S, D = x.shape
            x_flat = x.reshape(-1, D)
        else:
            x_flat = x

        # ---- 路由 ----
        logits = self.router(x_flat)  # (N, num_experts)
        top_vals, top_idx = logits.topk(self.top_k, dim=-1)
        weights = torch.softmax(top_vals, dim=-1)  # (N, top_k)

        # ---- 手写加权组合 ----
        # 面试可能要求不用循环，用 scatter 操作
        # 但循环版本更清晰，面试中先写循环再优化
        output = torch.zeros_like(x_flat)
        for k in range(self.top_k):
            for e in range(len(self.experts)):
                mask = (top_idx[:, k] == e)
                if mask.any():
                    output[mask] += weights[mask, k:k+1] * self.experts[e](x_flat[mask])

        return output.reshape(orig_shape)

# 面试核心追问：
# Q: MoE 的计算优势？
# A: 虽然总参数量很大（num_experts × 单个专家参数），
#    但每个 token 只激活 top_k 个专家，计算量 = top_k/num_experts × 全部
#    例如 8 个专家 top_2：每个 token 只用 25% 的参数
# Q: 路由不均衡（专家负载不均）怎么办？
# A: 1) 辅助损失（auxiliary loss）鼓励均匀路由
#    2) 噪声注入到 logits 增加探索
#    3) 容量因子（capacity factor）限制每个专家处理的 token 数
# Q: 为什么 top_k 而不是 1？
# A: top_1 路由太"硬"，训练不稳定。
#    top_k 让多个专家协作，输出更平滑。


In [ ]:
moe = MixtureOfExperts(d_model=64, d_ff=128, num_experts=4, top_k=2)
x = torch.randn(2, 10, 64)
out = moe(x)
print(f'Output: {out.shape}')
print(f'Total params: {sum(p.numel() for p in moe.parameters())}')


In [ ]:
from torch_judge import check
check('moe')


## 📝 核心思路总结

1. **路由器**：Linear(d, num_experts) + topk + softmax 决定每个 token 去哪些专家
2. **专家网络**：每个专家是独立的 FFN（Linear→ReLU→Linear）
3. **条件计算**：每个 token 只激活 top_k 个专家，计算量与专家数无关
4. **加权组合**：softmax 归一化后的权重 × 专家输出，求和得到最终结果
